# Phase 2 — Optimization v3
GPU experiments, validation selection and candidate runtime. The existing release remains available. Stretch goals above 90% are evaluated from actual results; training does not guarantee them. All stages save to Drive.


## 1. Setup

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, urllib.request, hashlib, json, torch

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before starting.'
print('GPU:', torch.cuda.get_device_name(0))
CODE = Path('/content/phase2_v3_code'); CODE.mkdir(exist_ok=True)
CHECKS = {'optimize.py': '7624974ae0d83c56e82ccd24d025353f47f7dd10a4893fa42933f11a3660ecb3', 'runtime.py': '206c5942b71ad61edba5491ca2cbbbe0220837063b0441cc33512167e26e70e0', 'requirements.txt': '71761a0f3267f81f9eda13cc4f28203aca511ec0439ae0fd0ed7acf84076cae8'}
for name, expected in CHECKS.items():
    url = 'https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/experiments/phase2_v3/' + name
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected, name
    (CODE/name).write_bytes(data)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE/'requirements.txt')], check=True)
CACHE = Path('/content/phase2_v3_cache')
RUN = ROOT/'phase2_results/optimization_v3'; RUN.mkdir(parents=True, exist_ok=True)
identity = RUN/'source_checksums.json'
if identity.exists():
    assert json.loads(identity.read_text()) == CHECKS, 'Source version changed. Preserve this run and use a new version.'
else:
    identity.write_text(json.dumps(CHECKS, indent=2))
def run_stage(stage):
    subprocess.run([sys.executable, '-u', str(CODE/'optimize.py'), '--root', str(ROOT), '--cache', str(CACHE), '--stage', stage], check=True)
print('Ready:', RUN)


## 2. Prepare data
Verify original images, copy to fast local storage and audit reference masks. Test images are excluded.

In [ ]:
run_stage('prepare')


## 3. Fine-tune detector
Larger YOLO11s at 960 pixels; up to 70 epochs with early stopping. Rerun to resume.

In [ ]:
run_stage('detection')


## 4. Fine-tune segmentation
YOLO11s-seg trained on reviewed masks; direct masks are a candidate for reducing SAM latency.

In [ ]:
run_stage('segmentation')


## 5. Fine-tune and calibrate species
Fine-tune the last ResNet stage with class-weighted loss. Select on validation macro-F1; calibrate separately.

In [ ]:
run_stage('species')


## 6. Compare and freeze candidates
Compare real validation metrics with baseline checkpoints. Thresholds are selected here; test is excluded.

In [ ]:
run_stage('select')


## 7. Export measured validation summary
Write only measured results. Deployment quality/latency and independent evaluation have separate scopes.

In [ ]:
run_stage('report')


## 8. Candidate HTTP demo and CPU benchmark
Real five-image HTTP request, overlays and 20 full-pipeline CPU measurements. CPU inference can take several minutes. This is a candidate measurement, not automatic replacement of v2.

In [ ]:
subprocess.run([sys.executable, '-u', str(CODE/'runtime.py'), '--root', str(ROOT), '--cache', str(CACHE), '--device', 'cpu'], check=True)


## 9. Export results for review
The package includes logs, policies, plots and receipts. Learned checkpoints remain on Drive; keep a separate model backup.

In [ ]:
import zipfile
archive = RUN/'Optimization_V3_Review.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RUN.rglob('*')):
        if path.is_file() and path.suffix in ('.json', '.csv', '.md', '.png', '.jpg', '.yaml'):
            z.write(path, path.relative_to(RUN))
print('Review package:', archive)
from google.colab import files
files.download(str(archive))
